# AI Meeting Notes & Action Items Agent (final, Groq)
Run top to bottom, one cell at a time. Groq free tier: ~200k tokens/day, so start small.

## 1. Install

In [ ]:
!pip install -q langgraph groq jsonschema python-dotenv requests pandas numpy scipy

## 2. API key (Secrets: GROQ_API_KEY, Notebook access ON)

In [ ]:
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY').strip()
print(os.environ['GROQ_API_KEY'][:4])

## 3. Upload data (audio.jsonl, text.jsonl, image.jsonl)

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('data', exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, f'data/{name}')
print(os.listdir('data'))

## 4. Project files

In [ ]:
%%writefile loader.py
import json
from pathlib import Path

def load(modality, limit=None, offset=0):
    rows = []
    with open(Path("data") / f"{modality}.jsonl", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i < offset:
                continue
            if limit is not None and len(rows) >= limit:
                break
            r = json.loads(line)
            r["json_schema"] = json.loads(r["json_schema"])
            r["ground_truth"] = json.loads(r["ground_truth"])
            r["modality"] = modality
            rows.append(r)
    return rows

In [ ]:
%%writefile preprocess.py
import re, collections

LINE = re.compile(r"^\[(\d+:\d+)\]\s+([A-Z]{3}\d{3}):\s*(.*)$")
FILLERS = {"mm", "hmm", "mm-hmm", "mm-hm", "uh-huh", "mhm", "um", "uh", "ah", "oh", "hm"}

def clean_audio(text):
    turns, words = [], collections.Counter()
    for line in text.splitlines():
        m = LINE.match(line.strip())
        if not m:
            continue
        t, spk, utt = m.groups()
        toks = utt.lower().split()
        if not toks or all(w.strip(".,'?") in FILLERS for w in toks):
            continue                      # drop pure backchannel like "MM", "HMM"
        words[spk] += len(toks)
        if turns and turns[-1][1] == spk:
            turns[-1][2].extend(toks)     # merge consecutive lines of same speaker
        else:
            turns.append([t, spk, toks])
    out = "\n".join(f"[{t}] {s}: {' '.join(w)}" for t, s, w in turns)
    total = sum(words.values()) or 1
    stats = {s: {"words": c, "share": round(c / total, 3)} for s, c in words.items()}
    return out, stats

def clean_ocr(text):
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # re-join hyphenated line breaks
    return re.sub(r"[ \t]+", " ", text).strip(), {}

def clean_text(text):
    return re.sub(r"\n{3,}", "\n\n", text).strip(), {}

def run(modality, text):
    return {"audio": clean_audio, "image": clean_ocr}.get(modality, clean_text)(text)

In [ ]:
%%writefile evaluate.py
import re, unicodedata
import numpy as np
from scipy.optimize import linear_sum_assignment

def _tokens(x):
    s = unicodedata.normalize("NFKC", str(x)).lower()
    return set(re.findall(r"\w+", s))      # hyphens, unicode dashes, punctuation all split words

def leaf_score(p, g):
    if isinstance(g, (int, float, bool)) or g is None:
        return 1.0 if p == g else 0.0
    tp, tg = _tokens(p), _tokens(g)
    if not tp or not tg:
        return float(tp == tg)
    inter = len(tp & tg)
    if inter == 0:
        return 0.0
    pr, rc = inter / len(tp), inter / len(tg)
    return 2 * pr * rc / (pr + rc)

def score(p, g):
    if isinstance(g, dict):
        if not isinstance(p, dict):
            return 0.0
        return sum(score(p.get(k), v) for k, v in g.items()) / max(len(g), 1)
    if isinstance(g, list):
        if not isinstance(p, list) or not g:
            return float(p == g)
        if not p:
            return 0.0
        m = [[score(pj, gi) for pj in p] for gi in g]          # gold x predicted score matrix
        rows, cols = linear_sum_assignment(-np.array(m))       # best one-to-one matching (order independent)
        return float(sum(m[i][j] for i, j in zip(rows, cols))) / max(len(g), len(p))
    return leaf_score(p, g)

def top_level_scores(p, g):
    if not isinstance(g, dict):
        return {}
    p = p if isinstance(p, dict) else {}
    return {k: round(score(p.get(k), v), 3) for k, v in g.items()}

In [ ]:
%%writefile db.py
import sqlite3, json, re

def conn():
    return sqlite3.connect("meetings.db", timeout=30)

def init_db():
    c = conn()
    c.executescript("""
    CREATE TABLE IF NOT EXISTS runs(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        experiment TEXT, record_id TEXT, modality TEXT, question TEXT,
        prediction TEXT, schema_valid INTEGER, grounded INTEGER,
        score REAL, field_scores TEXT, attempts INTEGER,
        input_tokens INTEGER, output_tokens INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);
    CREATE TABLE IF NOT EXISTS action_items(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id INTEGER, assignee TEXT, task TEXT, due TEXT,
        status TEXT DEFAULT 'pending',
        FOREIGN KEY(run_id) REFERENCES runs(id));
    CREATE TABLE IF NOT EXISTS speaker_stats(
        run_id INTEGER, speaker TEXT, words INTEGER, share REAL,
        FOREIGN KEY(run_id) REFERENCES runs(id));
    """)
    c.commit(); c.close()

_A = re.compile(r"assign|owner|responsib")          # who
_NOT_ROLE = re.compile(r"role")
_DUE = re.compile(r"due|deadline|timeline")           # when
_TASK_PRIORITY = [re.compile(p) for p in (r"task", r"action", r"description", r"next_step")]

def _first(d, rx, exclude=None):
    for k, v in d.items():
        kl = k.lower()
        if rx.search(kl) and not (exclude and exclude.search(kl)) and isinstance(v, (str, int, float)) and v not in ("", None):
            return str(v)
    return None

def _task(d):
    for rx in _TASK_PRIORITY:
        for k, v in d.items():
            kl = k.lower()
            if rx.search(kl) and not _A.search(kl) and not _DUE.search(kl) and isinstance(v, str) and v:
                return v
    return None

def find_action_items(obj):
    # any list of dicts that has an assignee-like key AND a task-like key counts as action items
    out = []
    if isinstance(obj, dict):
        for v in obj.values():
            out += find_action_items(v)
    elif isinstance(obj, list):
        for it in obj:
            if isinstance(it, dict):
                a = _first(it, _A, _NOT_ROLE)
                t = _task(it)
                if a and t:
                    out.append((a, t, _first(it, _DUE)))
                    continue
            out += find_action_items(it)
    return out

def save_run(record, pred, valid, grounded, score, field_scores, attempts,
             in_tok, out_tok, experiment, stats):
    c = conn(); cur = c.cursor()
    cur.execute("""INSERT INTO runs(experiment,record_id,modality,question,prediction,schema_valid,
                   grounded,score,field_scores,attempts,input_tokens,output_tokens)
                   VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
                (experiment, record["record_id"], record["modality"], record["question"],
                 json.dumps(pred), int(valid), None if grounded is None else int(grounded),
                 score, json.dumps(field_scores), attempts, in_tok, out_tok))
    rid = cur.lastrowid
    for a, t, d in find_action_items(pred):
        cur.execute("INSERT INTO action_items(run_id,assignee,task,due) VALUES (?,?,?,?)", (rid, a, t, d))
    for spk, s in (stats or {}).items():
        cur.execute("INSERT INTO speaker_stats VALUES (?,?,?,?)", (rid, spk, s["words"], s["share"]))
    c.commit(); c.close()
    return rid

In [ ]:
%%writefile llm.py
import os, json, time, hashlib, threading, collections

MODEL = "openai/gpt-oss-120b"     # check console.groq.com/docs/models for current names
TPM = 7000                        # free tier ~8000 tokens/min. Paid tier: raise this (e.g. 100000)
MAX_DIRECT_CHARS = 18000          # longer contexts are condensed chunk-by-chunk first
CHUNK_CHARS = 14000

SYSTEM = ("You extract structured data from the given context. Use ONLY facts stated in the context; "
          "never invent values. For meeting transcripts, speakers appear as IDs like MEO069 - use those IDs "
          "as assignees. The transcript is noisy ASR text, so interpret it sensibly. "
          "Keep every string value short: one concise sentence, copying the speakers' own wording where possible. "
          "For action items include only tasks a speaker explicitly commits to or is assigned; set the deadline to "
          "the time phrase actually said (e.g. 'Wednesday afternoon'), never a guess. "
          "When asked about tasks per person, include one entry for EVERY person who is assigned something, "
          "and never merge different people's tasks into one entry. "
          "Return ONLY a JSON object that matches the required schema.")

CRITIC_SCHEMA = {"type": "object",
                 "properties": {"grounded": {"type": "boolean"},
                                "issues": {"type": "array", "items": {"type": "string"}}},
                 "required": ["grounded", "issues"]}

_client = None
_lock = threading.Lock()
_window = collections.deque()     # [timestamp, tokens] of calls in the last 60s
_notes_cache = {}

def client():
    global _client
    if _client is None:
        from groq import Groq
        _client = Groq(max_retries=5)      # reads GROQ_API_KEY from environment
    return _client

def _throttle(est):
    with _lock:
        while True:
            now = time.time()
            while _window and now - _window[0][0] > 60:
                _window.popleft()
            used = sum(t for _, t in _window)
            if used + est <= TPM or not _window:
                break
            wait = max(60 - (now - _window[0][0]) + 0.5, 1)
            print(f"   (rate limit: waiting {wait:.0f}s)")
            time.sleep(wait)
        entry = [time.time(), est]
        _window.append(entry)
        return entry

def _chat(system, user, max_out, response_format=None):
    est = (len(system) + len(user)) // 4 + min(max_out, 1500)
    entry = _throttle(est)
    kw = dict(model=MODEL, temperature=0, max_completion_tokens=max_out,
              messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    if MODEL.startswith("openai/gpt-oss"):
        kw["reasoning_effort"] = "low"
    if response_format:
        kw["response_format"] = response_format
    resp = client().chat.completions.create(**kw)
    u = (resp.usage.prompt_tokens, resp.usage.completion_tokens)
    entry[1] = u[0] + u[1]                  # replace estimate with real usage
    return resp.choices[0].message.content or "", u

def _parse(text):
    try:
        return json.loads(text)
    except Exception:
        a, b = text.find("{"), text.rfind("}")
        if a != -1 and b > a:
            try:
                return json.loads(text[a:b + 1])
            except Exception:
                return None
    return None

def _chat_json(system, user, schema, max_out):
    rf = {"type": "json_schema", "json_schema": {"name": "submit", "schema": schema}}
    try:
        text, u = _chat(system, user, max_out, rf)
    except Exception as e:
        if getattr(e, "status_code", None) != 400:
            raise
        # model/schema not supported by json_schema mode -> fall back to plain JSON mode
        user2 = user + "\n\nRequired JSON schema:\n" + json.dumps(schema)
        text, u = _chat(system, user2, max_out, {"type": "json_object"})
    return _parse(text), u

def _schema_hint(schema, prefix="", out=None):
    out = [] if out is None else out
    for k, v in ((schema or {}).get("properties") or {}).items():
        d = v.get("description", "")
        if v.get("type") == "object":
            _schema_hint(v, prefix + k + ".", out)
        elif v.get("type") == "array" and isinstance(v.get("items"), dict) and v["items"].get("type") == "object":
            out.append(f"- {prefix}{k}[] (list): {d}")
            _schema_hint(v["items"], prefix + k + "[].", out)
        else:
            out.append(f"- {prefix}{k}: {d}")
    return out

def _condense(context, question, schema=None):
    """Long transcripts: write notes per chunk (map), then answer from the notes (reduce)."""
    if len(context) <= MAX_DIRECT_CHARS:
        return context, (0, 0)
    key = hashlib.md5((context + question).encode()).hexdigest()
    if key in _notes_cache:
        return _notes_cache[key], (0, 0)
    chunks, cur = [], ""
    for line in context.splitlines(keepends=True):
        if len(cur) + len(line) > CHUNK_CHARS and cur:
            chunks.append(cur); cur = ""
        cur += line
    if cur:
        chunks.append(cur)
    hint = "\n".join(_schema_hint(schema))[:2500]
    hint_txt = f"\nThe final answer must fill these fields, so capture facts for them:\n{hint}\n" if hint else ""
    notes, tin, tout = [], 0, 0
    sysmsg = "You take careful notes on meeting transcripts."
    for i, ch in enumerate(chunks, 1):
        user = (f"Question to answer later: {question}\n{hint_txt}\nTranscript part {i} of {len(chunks)}:\n{ch}\n\n"
                "Write notes (max 450 words) relevant to the question: decisions, key facts and numbers, "
                "and action items (who - use speaker IDs, what, deadline). If nothing is relevant, write 'nothing relevant'.")
        text, u = _chat(sysmsg, user, 1100)
        notes.append(f"[Notes, part {i}/{len(chunks)}]\n{text.strip()}")
        tin += u[0]; tout += u[1]
    condensed = "\n\n".join(notes)
    _notes_cache[key] = condensed
    return condensed, (tin, tout)

def call_extract(context, question, schema, feedback=None):
    ctx, (ti, to) = _condense(context, question, schema)
    user = f"<context>\n{ctx}\n</context>\n\nQuestion: {question}"
    if feedback:
        user += f"\n\nYour previous answer had a problem: {feedback}\nFix it and answer again."
    pred, u = _chat_json(SYSTEM, user, schema, 6000)
    return pred, (u[0] + ti, u[1] + to)

def call_critic(context, question, prediction):
    ctx, (ti, to) = _condense(context, question)
    user = (f"<context>\n{ctx}\n</context>\n\nQuestion: {question}\n\nProposed answer (JSON):\n{prediction}\n\n"
            "Check every value in the answer against the context. List each value that is NOT supported by "
            "the context or contradicts it. If everything is supported, set grounded=true and issues=[].")
    v, u = _chat_json("You are a strict fact-checker. Return ONLY JSON.", user, CRITIC_SCHEMA, 1500)
    v = v if isinstance(v, dict) and "grounded" in v else {"grounded": True, "issues": []}
    return v, (u[0] + ti, u[1] + to)

In [ ]:
%%writefile agent.py
import os, requests, jsonschema
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt
import llm, db, preprocess
from evaluate import score as score_fn, top_level_scores

MAX_ATTEMPTS = 3

class State(TypedDict):
    record: dict
    options: dict
    context: str
    stats: dict
    prediction: Optional[dict]
    feedback: Optional[str]
    attempts: int
    valid: bool
    grounded: Optional[bool]
    score: Optional[float]
    field_scores: Optional[dict]
    in_tok: int
    out_tok: int
    rejected: bool
    run_id: Optional[int]

# ---------- nodes ----------
def prep(state: State):
    r = state["record"]
    if state["options"].get("preprocess", True):
        ctx, stats = preprocess.run(r["modality"], r["context"])
    else:
        ctx, stats = r["context"], {}
    return {"context": ctx, "stats": stats}

def extract(state: State):
    pred, u = llm.call_extract(state["context"], state["record"]["question"],
                               state["record"]["json_schema"], state["feedback"])
    return {"prediction": pred, "attempts": state["attempts"] + 1,
            "in_tok": state["in_tok"] + u[0], "out_tok": state["out_tok"] + u[1]}

def validate(state: State):
    if state["prediction"] is None:
        return {"valid": False, "feedback": "No tool output returned (maybe truncated). Give a complete, concise answer."}
    try:
        jsonschema.validate(state["prediction"], state["record"]["json_schema"])
        return {"valid": True, "feedback": None}
    except jsonschema.ValidationError as e:
        path = "/".join(map(str, e.absolute_path))
        return {"valid": False, "feedback": f"Schema error at '{path}': {e.message[:250]}"}

def after_validate(state: State):
    if not state["valid"]:
        return "extract" if state["attempts"] < MAX_ATTEMPTS else "grade"
    return "critique" if state["options"].get("reflect") else "grade"

def critique(state: State):
    v, u = llm.call_critic(state["context"], state["record"]["question"], state["prediction"])
    out = {"grounded": v["grounded"], "in_tok": state["in_tok"] + u[0], "out_tok": state["out_tok"] + u[1]}
    if not v["grounded"]:
        out["feedback"] = "These parts are not supported by the context: " + "; ".join(v["issues"])
    return out

def after_critique(state: State):
    return "extract" if (not state["grounded"] and state["attempts"] < MAX_ATTEMPTS) else "grade"

def grade(state: State):
    gt = state["record"].get("ground_truth")
    if gt is None:
        return {"score": None, "field_scores": None}
    return {"score": score_fn(state["prediction"], gt),
            "field_scores": top_level_scores(state["prediction"], gt)}

def approve(state: State):
    if not state["options"].get("approve"):
        return {}
    answer = interrupt({"question": "Save this result? (yes/no)", "prediction": state["prediction"]})
    return {"rejected": str(answer).strip().lower() not in ("y", "yes")}

def after_approve(state: State):
    return END if state["rejected"] else "save"

def save(state: State):
    rid = db.save_run(state["record"], state["prediction"], state["valid"], state["grounded"],
                      state["score"], state["field_scores"], state["attempts"],
                      state["in_tok"], state["out_tok"],
                      state["options"].get("experiment", "baseline"), state["stats"])
    return {"run_id": rid}

def notify(state: State):
    n = len(db.find_action_items(state["prediction"]))
    msg = (f"[{state['options'].get('experiment')}] run {state['run_id']}: {n} action items, "
           f"score={None if state['score'] is None else round(state['score'], 3)}, attempts={state['attempts']}")
    hook = os.getenv("SLACK_WEBHOOK_URL")
    if hook:
        requests.post(hook, json={"text": msg}, timeout=10)
    print(msg)
    return {}

# ---------- graph ----------
def build(checkpointer=None):
    db.init_db()
    g = StateGraph(State)
    for name, fn in [("prep", prep), ("extract", extract), ("validate", validate), ("critique", critique),
                     ("grade", grade), ("approve", approve), ("save", save), ("notify", notify)]:
        g.add_node(name, fn)
    g.add_edge(START, "prep")
    g.add_edge("prep", "extract")
    g.add_edge("extract", "validate")
    g.add_conditional_edges("validate", after_validate, ["extract", "critique", "grade"])
    g.add_conditional_edges("critique", after_critique, ["extract", "grade"])
    g.add_edge("grade", "approve")
    g.add_conditional_edges("approve", after_approve, ["save", END])
    g.add_edge("save", "notify")
    g.add_edge("notify", END)
    return g.compile(checkpointer=checkpointer)

DEFAULTS = {"experiment": "baseline", "preprocess": True, "reflect": False, "approve": False}

def run_record(graph, record, options=None, thread_id=None):
    init = {"record": record, "options": {**DEFAULTS, **(options or {})}, "context": "", "stats": {},
            "prediction": None, "feedback": None, "attempts": 0, "valid": False, "grounded": None,
            "score": None, "field_scores": None, "in_tok": 0, "out_tok": 0, "rejected": False, "run_id": None}
    cfg = {"configurable": {"thread_id": thread_id or record["record_id"]}}
    return graph.invoke(init, cfg)

In [ ]:
%%writefile batch.py
import json, pandas as pd
from concurrent.futures import ThreadPoolExecutor
import agent, db
from loader import load

def run_batch(modality, n=10, workers=4, offset=0, **options):
    graph = agent.build()
    recs = load(modality, limit=n, offset=offset)
    def one(r):
        try:
            return agent.run_record(graph, r, options)
        except Exception as e:
            print("FAILED", r["record_id"][:8], repr(e)[:150])
            return None
    with ThreadPoolExecutor(workers) as ex:
        outs = [o for o in ex.map(one, recs) if o]
    k = max(len(outs), 1)
    summary = {"experiment": options.get("experiment", "baseline"), "modality": modality, "done": len(outs),
               "failed": len(recs) - len(outs),
               "avg_score": round(sum(o["score"] or 0 for o in outs) / k, 3),
               "valid_rate": round(sum(o["valid"] for o in outs) / k, 2),
               "avg_attempts": round(sum(o["attempts"] for o in outs) / k, 2),
               "avg_in_tok": int(sum(o["in_tok"] for o in outs) / k),
               "avg_out_tok": int(sum(o["out_tok"] for o in outs) / k)}
    print("\nSUMMARY:", summary)
    return summary

def compare():
    return pd.read_sql("""SELECT experiment, modality, COUNT(*) AS n, ROUND(AVG(score),3) AS avg_score,
        ROUND(AVG(schema_valid),2) AS valid, ROUND(AVG(attempts),2) AS attempts,
        CAST(AVG(input_tokens) AS INT) AS in_tok, CAST(AVG(output_tokens) AS INT) AS out_tok
        FROM runs GROUP BY experiment, modality ORDER BY modality, avg_score DESC""", db.conn())

def weak_fields(experiment="baseline", modality="audio", min_count=2):
    rows = db.conn().execute("SELECT field_scores FROM runs WHERE experiment=? AND modality=?",
                             (experiment, modality)).fetchall()
    agg = {}
    for (fs,) in rows:
        for k, v in json.loads(fs or "{}").items():
            agg.setdefault(k, []).append(v)
    df = pd.DataFrame([{"field": k, "count": len(v), "avg_score": round(sum(v) / len(v), 3)}
                       for k, v in agg.items() if len(v) >= min_count])
    return df.sort_values("avg_score").head(15) if len(df) else df

## 5. Key test

In [ ]:
import llm
print(llm.MODEL)
print(llm._chat('Be brief.', 'Say hi', 50))

## 6. Text run (small)

In [ ]:
import batch
batch.run_batch('text', n=5, workers=1, experiment='text_baseline')

## 7. Audio run on meetings that fit one request (cheaper, more accurate)

In [ ]:
import preprocess, agent
from loader import load
short = [r for r in load('audio') if 3000 < len(preprocess.run('audio', r['context'])[0]) <= 18000]
print(len(short), 'meetings fit directly')
graph = agent.build()
scores = []
for r in short[:3]:
    out = agent.run_record(graph, r, {'experiment': 'audio_direct'})
    scores.append(out['score'])
print('avg score:', round(sum(scores)/len(scores), 3))

## 8. Results

In [ ]:
import pandas as pd, db
display(batch.compare())
c = db.conn()
display(pd.read_sql("SELECT assignee, task, due FROM action_items LIMIT 15", c))

## 9. Save your results (Colab storage is temporary)

In [ ]:
from google.colab import files
files.download('meetings.db')